# PathSnap – Model training (Person 1)
Candidate classifier: for each (GPS point, candidate road arc) predict P(arc is the true arc).
All models use the **same** 17 features, the **same** 5-fold GroupKFold by trip (seed 42) and the **same** `evaluate()` function.

## 0. Setup
Upload `pathsnap_person1.zip` and the course `Student folder` (zipped as `Student_folder.zip`) to Colab, then run:

In [ ]:
!pip -q install lightgbm xgboost optuna pyproj folium
!unzip -q -o pathsnap_person1.zip
!unzip -q -o Student_folder.zip -d data_raw
import os, sys
os.environ["PATHSNAP_DATA"] = "/content/data_raw/Student folder"
os.chdir("/content/pathsnap"); sys.path.insert(0, "src")

## 1. Build the dataset (candidates + features + labels + folds)
R = 50 m radius, nearest K = 30 candidates, labels from DP alignment of GPS points to the route. ~1 min.

In [ ]:
!python src/build_dataset.py --R 50 --K 30

In [ ]:
import pandas as pd
F = pd.read_parquet("data/processed/features.parquet")
print(F.shape); print(F.groupby("partition").label.agg(["size", "mean"]))
F.head()

## 2. Shared evaluator
`evaluate(model, name, family, config)` runs 5-fold GroupKFold CV on the 80 train trips, refits on all of them, then scores the 10 student-test trips (Top-1, macro F1, precision, recall, log loss, route continuity, timings) and logs a row to `results/results.csv`.

In [ ]:
from evaluate import evaluate, NearestArcBaseline, FEATURES
from common import SEED
print(len(FEATURES), "features:", FEATURES)

## 3. Models

### Reference: Nearest-arc (no ML)
Picks the closest arc. Shows what learning adds over pure geometry.

In [ ]:
model = NearestArcBaseline()
evaluate(model, name="nearest_arc", family="Heuristic", config={"rule": "nearest arc"})

**Recorded results (from `results/results.csv`):** CV macro F1 = 0.7508 ± 0.0099 | Train macro F1 = 0.7527 | Student-test Top-1 = 0.5442 | Test macro F1 = 0.7475 | Precision = 0.511 | Recall = 0.588 | Log loss = 0.2573 | Route continuity = 0.820 | Train time = 0.1s | Inference = 0.1 ms/1k pts

### Model 1: Logistic Regression (baseline, linear)
Scaled features → L2 logistic regression. Calibrated probabilities, interpretable, but no feature interactions.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = make_pipeline(StandardScaler(),
                      LogisticRegression(C=1.0, max_iter=1000, random_state=SEED))
evaluate(model, name="logreg", family="Linear / probabilistic",
         config={"C": 1.0, "penalty": "l2", "solver": "lbfgs"})

**Recorded results (from `results/results.csv`):** CV macro F1 = 0.9013 ± 0.0139 | Train macro F1 = 0.9053 | Student-test Top-1 = 0.8695 | Test macro F1 = 0.9118 | Precision = 0.845 | Recall = 0.836 | Log loss = 0.0869 | Route continuity = 0.931 | Train time = 4.3s | Inference = 0.9 ms/1k pts

### Model 2: Gaussian Naive Bayes (generative)
Quantile→normal transform so features look Gaussian; NB assumes independence (violated by our correlated distance features → over-confident).

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import QuantileTransformer
from sklearn.naive_bayes import GaussianNB

model = make_pipeline(QuantileTransformer(output_distribution="normal", n_quantiles=1000,
                                          subsample=200_000, random_state=SEED),
                      GaussianNB())
evaluate(model, name="gaussian_nb", family="Probabilistic (generative)",
         config={"transform": "QuantileTransformer(normal)", "var_smoothing": 1e-9})

**Recorded results (from `results/results.csv`):** CV macro F1 = 0.8055 ± 0.0104 | Train macro F1 = 0.8054 | Student-test Top-1 = 0.8388 | Test macro F1 = 0.8066 | Precision = 0.525 | Recall = 0.902 | Log loss = 0.6234 | Route continuity = 0.896 | Train time = 4.1s | Inference = 18.6 ms/1k pts

### Model 3: Random Forest (bagging ensemble)
Non-linear, robust; bootstrap fraction 0.2 to keep training time reasonable on 1.77M rows (~8 min with CV).

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, min_samples_leaf=5, max_features="sqrt",
                               max_samples=0.2, n_jobs=-1, random_state=SEED)
evaluate(model, name="random_forest", family="Ensemble (bagging)",
         config={"n_estimators": 100, "min_samples_leaf": 5, "max_samples": 0.2})

**Recorded results (from `results/results.csv`):** CV macro F1 = 0.9235 ± 0.0158 | Train macro F1 = 0.9421 | Student-test Top-1 = 0.8776 | Test macro F1 = 0.9297 | Precision = 0.898 | Recall = 0.848 | Log loss = 0.0602 | Route continuity = 0.934 | Train time = 98.0s | Inference = 43.9 ms/1k pts

### Model 4: XGBoost (gradient boosting, depth-wise)

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=7, min_child_weight=5,
                      subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, tree_method="hist",
                      eval_metric="logloss", random_state=SEED, n_jobs=-1)
evaluate(model, name="xgboost", family="Boosting",
         config={"n_estimators": 400, "learning_rate": 0.05, "max_depth": 7})

**Recorded results (from `results/results.csv`):** CV macro F1 = 0.9256 ± 0.0155 | Train macro F1 = 0.9481 | Student-test Top-1 = 0.8831 | Test macro F1 = 0.9328 | Precision = 0.892 | Recall = 0.865 | Log loss = 0.0572 | Route continuity = 0.941 | Train time = 33.0s | Inference = 30.9 ms/1k pts

### Model 5: LightGBM (gradient boosting, leaf-wise), selected for tuning

In [ ]:
from lightgbm import LGBMClassifier

model = LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63, min_child_samples=50,
                       subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
                       random_state=SEED, n_jobs=-1, verbose=-1)
evaluate(model, name="lightgbm", family="Boosting",
         config={"n_estimators": 400, "learning_rate": 0.05, "num_leaves": 63})

**Recorded results (from `results/results.csv`):** CV macro F1 = 0.9262 ± 0.0158 | Train macro F1 = 0.9507 | Student-test Top-1 = 0.8814 | Test macro F1 = 0.9311 | Precision = 0.889 | Recall = 0.862 | Log loss = 0.0580 | Route continuity = 0.939 | Train time = 33.4s | Inference = 129.1 ms/1k pts

## 4. Bayesian optimisation (Optuna TPE) of LightGBM
Objective = mean macro F1 over the same GroupKFold on the 80 training trips. Search took ~47 min on 2 cores, so the cell below is set to a short demo budget. The full run used `--trials 40 --timeout 2700`.

In [ ]:
!python src/optuna_tune.py --model lightgbm --trials 5 --timeout 600

**Recorded full search:** 21 trials (13 complete, 8 pruned), best CV macro F1 = 0.9267, best params = `{"n_estimators": 450, "learning_rate": 0.0394, "num_leaves": 76, "min_child_samples": 59, "subsample": 0.9967, "colsample_bytree": 0.6939, "reg_lambda": 0.0303, "reg_alpha": 3.7945}`

**Recorded results (from `results/results.csv`):** CV macro F1 = 0.9267 ± 0.0154 | Train macro F1 = 0.9507 | Student-test Top-1 = 0.8832 | Test macro F1 = 0.9323 | Precision = 0.893 | Recall = 0.862 | Log loss = 0.0575 | Route continuity = 0.940 | Train time = 42.6s | Inference = 181.6 ms/1k pts

In [ ]:
import json
print(json.dumps(json.load(open("results/optuna/lightgbm_best.json")), indent=2))
pd.read_csv("results/optuna/tuned_vs_untuned.csv")

## 5. Comparison table + plots

In [ ]:
!python src/plots.py
cols = ["cv_macro_f1_mean","cv_macro_f1_std","test_top1","test_macro_f1","test_log_loss","test_route_continuity","train_time_s","inference_ms_per_1k_points"]
pd.read_csv("results/results.csv").set_index("model")[cols].round(4).sort_values("cv_macro_f1_mean", ascending=False)

In [ ]:
from IPython.display import Image, display
for p in ["summary_model_comparison","macro_f1_comparison","top1_comparison","logloss_inference","cv_folds","confusion_matrices","feature_importance_lightgbm"]:
    display(Image(f"results/plots/{p}.png"))

## 6. Final model export + map
Exports `artifacts/final_model.joblib` for the API, checks that the inference path reproduces the evaluation probabilities, and draws a Folium map of predictions.

In [ ]:
!python src/finalize.py --name lightgbm_tuned
from IPython.display import IFrame
IFrame("results/maps/student_test_00000077_predictions.html", width=1000, height=600)